# NB46 — iPhone T22 QA
Tower 22: Tower of iPhone — iOS/Safari web compatibility, Apple PWA support (apple-touch-icon, apple-mobile-web-app-capable), Capacitor iOS config, responsive design, safe area CSS, touch targets, and service worker.

**Status:** SETUP_ONLY — no ios/ directory exists yet. Probes focus on web-layer iOS readiness.

**Scope:** File-based probes only. No servers, no HTTP requests, no module imports.

In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime
NORTHSTAR = "iphone-t22-qa"
NOTEBOOK_PATH = "notebooks/qa/46-iphone-t22-qa.ipynb"
TOWER = 22
TOWER_NAME = "Tower of iPhone"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
results = []
def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))
def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""
print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 22: Tower of iPhone


In [2]:
# === Cell 1: Capacitor iOS config readiness ===

# P01: @capacitor/ios listed in package.json
pkg_json = read_file(BROWSER_ROOT / "package.json")
record("T22-P01", "@capacitor/ios dependency in package.json",
       '@capacitor/ios' in pkg_json)

# P02: capacitor config has ios section (.ts or .json)
cap_config_ts = read_file(BROWSER_ROOT / "capacitor.config.ts")
cap_config_json = read_file(BROWSER_ROOT / "capacitor.config.json")
cap_config = cap_config_ts if cap_config_ts else cap_config_json
cap_file = "capacitor.config.ts" if cap_config_ts else "capacitor.config.json"
# JSON format: "ios": { ... }, TS format: ios: { ... }
has_ios_section = ('"ios"' in cap_config or "'ios'" in cap_config or
                   'ios:' in cap_config or '"ios":' in cap_config)
record("T22-P02", "Capacitor config has ios config section",
       has_ios_section,
       f"ios section found in {cap_file}")

# P03: capacitor ios contentInset configured
record("T22-P03", "iOS contentInset configured in capacitor config",
       'contentInset' in cap_config,
       f"contentInset found in {cap_file}")

# P04: capacitor ios scheme configured
record("T22-P04", "iOS scheme configured in capacitor config",
       'scheme' in cap_config and ('Solace' in cap_config),
       f"iOS scheme found in {cap_file}")

PASS: @capacitor/ios dependency in package.json
PASS: Capacitor config has ios config section -- ios section found in capacitor.config.json
PASS: iOS contentInset configured in capacitor config -- contentInset found in capacitor.config.json
PASS: iOS scheme configured in capacitor config -- iOS scheme found in capacitor.config.json


In [3]:
# === Cell 2: Apple PWA meta tags ===

# Collect all main HTML files (excluding partials)
html_files = sorted([f for f in WEB.glob('*.html') if 'partial' not in f.name])
# Key pages that users visit directly
key_pages = ['home.html', 'app-store.html', 'settings.html', 'schedule.html',
             'start.html', 'guide.html', 'download.html', 'demo.html']

# P05: apple-touch-icon on all key pages
pages_with_ati = 0
for page in key_pages:
    content = read_file(WEB / page)
    if 'apple-touch-icon' in content:
        pages_with_ati += 1
record("T22-P05", "apple-touch-icon on all key HTML pages",
       pages_with_ati == len(key_pages),
       f"{pages_with_ati}/{len(key_pages)} pages")

# P06: apple-mobile-web-app-capable on at least start.html
start_html = read_file(WEB / "start.html")
record("T22-P06", "apple-mobile-web-app-capable on start.html",
       'apple-mobile-web-app-capable' in start_html)

# P07: apple-mobile-web-app-status-bar-style set
record("T22-P07", "apple-mobile-web-app-status-bar-style set",
       'apple-mobile-web-app-status-bar-style' in start_html)

# P08: apple-mobile-web-app-title set
record("T22-P08", "apple-mobile-web-app-title set on start.html",
       'apple-mobile-web-app-title' in start_html)

# P09: theme-color meta tag on >= 80% of key pages
pages_with_theme = 0
for page in key_pages:
    content = read_file(WEB / page)
    if 'theme-color' in content:
        pages_with_theme += 1
theme_ratio = pages_with_theme / len(key_pages) if key_pages else 0
record("T22-P09", "theme-color meta on >= 80% of key pages",
       theme_ratio >= 0.80,
       f"{pages_with_theme}/{len(key_pages)} pages ({theme_ratio:.0%})")

PASS: apple-touch-icon on all key HTML pages -- 8/8 pages
PASS: apple-mobile-web-app-capable on start.html
PASS: apple-mobile-web-app-status-bar-style set
PASS: apple-mobile-web-app-title set on start.html
PASS: theme-color meta on >= 80% of key pages -- 7/8 pages (88%)


In [4]:
# === Cell 3: PWA manifest iOS readiness ===

# P10: manifest.json exists and is valid
manifest_raw = read_file(WEB / "manifest.json")
has_manifest = len(manifest_raw) > 0
mdata = json.loads(manifest_raw) if has_manifest else {}
record("T22-P10", "manifest.json exists and is valid JSON",
       has_manifest and bool(mdata))

# P11: manifest display = standalone (required for iOS PWA)
record("T22-P11", "manifest display = standalone",
       mdata.get('display') == 'standalone')

# P12: manifest has start_url
record("T22-P12", "manifest has start_url",
       bool(mdata.get('start_url')))

# P13: manifest icons include 192 and 512
icon_sizes = [i.get('sizes', '') for i in mdata.get('icons', [])]
has_192 = any('192' in s for s in icon_sizes)
has_512 = any('512' in s for s in icon_sizes)
record("T22-P13", "manifest has 192x192 and 512x512 icons",
       has_192 and has_512,
       f"192={has_192}, 512={has_512}")

# P14: manifest has screenshots with narrow form_factor
screenshots = mdata.get('screenshots', [])
has_narrow = any(s.get('form_factor') == 'narrow' for s in screenshots)
record("T22-P14", "manifest has narrow (mobile) screenshot",
       has_narrow)

PASS: manifest.json exists and is valid JSON
PASS: manifest display = standalone
PASS: manifest has start_url
PASS: manifest has 192x192 and 512x512 icons -- 192=True, 512=True
PASS: manifest has narrow (mobile) screenshot


In [5]:
# === Cell 4: Service worker and offline support ===

# P15: sw.js exists with install/activate/fetch handlers
sw_content = read_file(WEB / "sw.js")
has_install = "addEventListener('install'" in sw_content or 'addEventListener("install"' in sw_content
has_activate = "addEventListener('activate'" in sw_content or 'addEventListener("activate"' in sw_content
has_fetch = "addEventListener('fetch'" in sw_content or 'addEventListener("fetch"' in sw_content
record("T22-P15", "sw.js has install/activate/fetch handlers",
       has_install and has_activate and has_fetch,
       f"install={has_install}, activate={has_activate}, fetch={has_fetch}")

# P16: sw.js has push notification handler
has_push = "addEventListener('push'" in sw_content or 'addEventListener("push"' in sw_content
record("T22-P16", "sw.js has push notification handler",
       has_push)

# P17: sw.js has notification click handler
has_notif_click = 'notificationclick' in sw_content
record("T22-P17", "sw.js has notificationclick handler",
       has_notif_click)

# P18: sw.js caches key assets for offline
has_cache_open = 'caches.open' in sw_content
has_precache = 'PRECACHE_URLS' in sw_content or 'cache.addAll' in sw_content
record("T22-P18", "sw.js precaches assets for offline",
       has_cache_open and has_precache)

PASS: sw.js has install/activate/fetch handlers -- install=True, activate=True, fetch=True
PASS: sw.js has push notification handler
PASS: sw.js has notificationclick handler
PASS: sw.js precaches assets for offline


In [6]:
# === Cell 5: Responsive design, touch targets, dark mode ===

site_css = read_file(WEB / "css" / "site.css")

# P19: CSS has mobile breakpoints for phone screens
mobile_bps = re.findall(r'@media\s*\(max-width:\s*(\d+)px\)', site_css)
mobile_bp_vals = sorted(set(int(v) for v in mobile_bps))
# iPhone SE: 375px, iPhone 14: 390px, iPhone Pro Max: 430px
has_small_bp = any(v <= 500 for v in mobile_bp_vals)
has_medium_bp = any(v <= 768 for v in mobile_bp_vals)
record("T22-P19", "CSS has phone-sized breakpoints",
       has_small_bp and has_medium_bp,
       f"breakpoints: {mobile_bp_vals}")

# P20: Touch targets >= 44px (Apple HIG minimum)
touch_targets = re.findall(r'min-(?:height|width):\s*4[4-9]px', site_css)
record("T22-P20", "Touch targets >= 44px in site.css",
       len(touch_targets) >= 2,
       f"{len(touch_targets)} occurrences")

# P21: prefers-color-scheme support (iOS dark/light mode)
record("T22-P21", "prefers-color-scheme media query in CSS",
       'prefers-color-scheme' in site_css)

# P22: Dark theme CSS file exists
dark_theme = (WEB / "css" / "themes" / "dark.css").exists()
light_theme = (WEB / "css" / "themes" / "light.css").exists()
record("T22-P22", "Dark and light theme CSS files exist",
       dark_theme and light_theme,
       f"dark={dark_theme}, light={light_theme}")

# P23: Viewport meta tag on all key pages
pages_with_vp = 0
for page in key_pages:
    content = read_file(WEB / page)
    if 'name="viewport"' in content:
        pages_with_vp += 1
record("T22-P23", "viewport meta tag on all key pages",
       pages_with_vp == len(key_pages),
       f"{pages_with_vp}/{len(key_pages)} pages")

# P24: manifest.json linked from key pages
pages_with_manifest = 0
for page in key_pages:
    content = read_file(WEB / page)
    if 'rel="manifest"' in content:
        pages_with_manifest += 1
record("T22-P24", "manifest.json linked from all key pages",
       pages_with_manifest == len(key_pages),
       f"{pages_with_manifest}/{len(key_pages)} pages")

PASS: CSS has phone-sized breakpoints -- breakpoints: [500, 540, 600, 680, 720, 767, 768, 900, 980]
PASS: Touch targets >= 44px in site.css -- 16 occurrences
PASS: prefers-color-scheme media query in CSS
PASS: Dark and light theme CSS files exist -- dark=True, light=True
PASS: viewport meta tag on all key pages -- 8/8 pages
PASS: manifest.json linked from all key pages -- 8/8 pages


In [7]:
# === Cell 6: iOS icon and PWA asset readiness ===

# P25: 256px apple-touch-icon image exists
ati_path = WEB / "images" / "yinyang" / "yinyang-logo-256.png"
record("T22-P25", "apple-touch-icon image (256px) exists on disk",
       ati_path.exists())

# P26: PWA icon-192.png exists
pwa_192 = WEB / "images" / "pwa" / "icon-192.png"
record("T22-P26", "PWA icon-192.png exists",
       pwa_192.exists())

# P27: PWA icon-512.png exists
pwa_512 = WEB / "images" / "pwa" / "icon-512.png"
record("T22-P27", "PWA icon-512.png exists",
       pwa_512.exists())

# P28: favicon.svg exists
record("T22-P28", "favicon.svg exists",
       (WEB / "favicon.svg").exists())

# P29: manifest orientation set (portrait-primary for iPhone)
record("T22-P29", "manifest orientation = portrait-primary",
       mdata.get('orientation') == 'portrait-primary')

# P30: manifest has shortcuts for quick actions
shortcuts = mdata.get('shortcuts', [])
record("T22-P30", "manifest has shortcuts for quick actions",
       len(shortcuts) >= 1,
       f"{len(shortcuts)} shortcuts defined")

PASS: apple-touch-icon image (256px) exists on disk
PASS: PWA icon-192.png exists
PASS: PWA icon-512.png exists
PASS: favicon.svg exists
PASS: manifest orientation = portrait-primary
PASS: manifest has shortcuts for quick actions -- 2 shortcuts defined


In [8]:
# === Evidence Summary ===
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 22: Tower of iPhone
Probes: 30 | Passed: 30 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: b98481c247d9bf91

{
  "surface": "iphone-t22-qa",
  "tower": 22,
  "tower_name": "Tower of iPhone",
  "timestamp": "2026-03-06T11:34:03.996486",
  "total_probes": 30,
  "passed": 30,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "b98481c247d9bf91"
}
